## swiGLU - Model retraining

This notebook is a continuation of the swiGLU workflows, it focuses on higher budget end-to-end retraining (KL loss) of the model, as prior selections used with tiny budgets.

The analyses focus on winning configuration from swiglu-2.

setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')


PROJECT_ROOT = find_project_root(Path.cwd())
SWIGLU_3_ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'workflows'
    / 'models'
    / 'swiglu-3'
    / 'run-001.json'
)

artifact = json.loads(
    SWIGLU_3_ARTIFACT_PATH.read_text(encoding='utf-8')
)
if artifact.get('status') != 'completed':
    raise ValueError('The SwiGLU-3 workflow artifact is not complete')

configuration = artifact['configuration']
results = artifact['results']
reference_path = (
    PROJECT_ROOT
    / configuration['references']['allocation_artifact']
)
reference_artifact = json.loads(
    reference_path.read_text(encoding='utf-8')
)
dense_reference = reference_artifact['results']['dense_reference'][0]

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', None)

In [ ]:
CALIBRATION_PAIR_ORDER = [
    int(value) for value in configuration['calibration']['pair_counts']
]
SPARSITY_ORDER = [
    float(value)
    for value in configuration['allocation']['target_mlp_removals']
]
SPARSITY_ORDER = sorted(SPARSITY_ORDER)
SPARSITY_LABEL_ORDER = [
    f'{value:.0%} MLP removal' for value in SPARSITY_ORDER
]
SPARSITY_PALETTE = dict(zip(
    SPARSITY_LABEL_ORDER,
    sns.color_palette('viridis', len(SPARSITY_LABEL_ORDER)),
))
DENSE_PARAMETERS = int(dense_reference['parameters'])
DENSE_PERPLEXITY = float(dense_reference['perplexity'])
SELECTED_CALIBRATION_PAIRS = int(
    results['calibration']['selected_calibration_pairs']
)

#### Calibration data sweep

In [ ]:
calibration_fitting_df = pd.DataFrame(
    results['calibration']['operator_fitting']
).sort_values(['calibration_pairs', 'layer'])
calibration_model_df = (
    pd.json_normalize(results['calibration']['model_evaluation'])
    .rename(columns={
        'allocation_selection.teacher_kl': 'selection_kl',
        'allocation_selection.loss': 'selection_loss',
        'allocation_selection.perplexity': 'selection_perplexity',
        'wikitext_validation.loss': 'wikitext_loss',
        'wikitext_validation.perplexity': 'wikitext_perplexity',
    })
)
calibration_local_summary_df = (
    calibration_fitting_df
    .groupby('calibration_pairs', as_index=False)
    .agg(
        fitted_blocks=('layer', 'size'),
        mean_initial_nmse=('initial_local_nmse', 'mean'),
        mean_nmse=('local_nmse', 'mean'),
        median_nmse=('local_nmse', 'median'),
        worst_nmse=('local_nmse', 'max'),
        mean_cosine=('local_cosine', 'mean'),
        mean_best_epoch=('best_epoch', 'mean'),
        total_updates=('updates', 'sum'),
        total_fit_minutes=(
            'fit_seconds', lambda values: values.sum() / 60
        ),
    )
)
calibration_summary_df = (
    calibration_local_summary_df
    .merge(
        calibration_model_df[[
            'calibration_pairs',
            'selection_kl',
            'selection_loss',
            'selection_perplexity',
            'wikitext_loss',
            'wikitext_perplexity',
        ]],
        on='calibration_pairs',
        validate='one_to_one',
    )
    .sort_values('calibration_pairs')
    .reset_index(drop=True)
)
calibration_summary_df['selected'] = (
    calibration_summary_df['calibration_pairs']
    == SELECTED_CALIBRATION_PAIRS
)

In [ ]:
display(
    calibration_summary_df.style.format({
        'calibration_pairs': '{:,.0f}',
        'mean_initial_nmse': '{:.5f}',
        'mean_nmse': '{:.5f}',
        'median_nmse': '{:.5f}',
        'worst_nmse': '{:.5f}',
        'mean_cosine': '{:.5f}',
        'mean_best_epoch': '{:.1f}',
        'total_updates': '{:,.0f}',
        'total_fit_minutes': '{:.1f}',
        'selection_kl': '{:.6f}',
        'selection_loss': '{:.5f}',
        'selection_perplexity': '{:.3f}',
        'wikitext_loss': '{:.5f}',
        'wikitext_perplexity': '{:.3f}',
    })
)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))

local_metric_df = calibration_summary_df.melt(
    id_vars='calibration_pairs',
    value_vars=['mean_nmse', 'worst_nmse'],
    var_name='statistic',
    value_name='nmse',
)
sns.lineplot(
    data=local_metric_df,
    x='calibration_pairs',
    y='nmse',
    hue='statistic',
    marker='o',
    ax=axes[0],
)
axes[0].set(
    title='Local fit versus calibration data',
    xlabel='Activation pairs per block',
    ylabel='Validation NMSE',
)

sns.lineplot(
    data=calibration_summary_df,
    x='calibration_pairs',
    y='selection_kl',
    marker='o',
    ax=axes[1],
)
axes[1].set(
    title='Integrated teacher KL',
    xlabel='Activation pairs per block',
    ylabel='C4 teacher KL',
)

sns.lineplot(
    data=calibration_summary_df,
    x='calibration_pairs',
    y='wikitext_perplexity',
    marker='o',
    ax=axes[2],
)
axes[2].axhline(
    DENSE_PERPLEXITY, color='black', linestyle='--', label='Dense model'
)
axes[2].set(
    title='End-to-end validation quality',
    xlabel='Activation pairs per block',
    ylabel='WikiText-2 perplexity',
)
axes[2].legend()

for axis in axes:
    axis.set_xscale('log', base=2)
    axis.set_xticks(
        CALIBRATION_PAIR_ORDER,
        [f'{value:,}' for value in CALIBRATION_PAIR_ORDER],
        rotation=20,
    )
figure.tight_layout()
plt.show()

In [ ]:
calibration_block_df = (
    calibration_fitting_df
    .pivot(index='layer', columns='calibration_pairs', values='local_nmse')
    .rename(columns=lambda value: f'nmse_{int(value)}')
    .reset_index()
)
baseline_pairs = min(CALIBRATION_PAIR_ORDER)
baseline_column = f'nmse_{baseline_pairs}'
selected_column = f'nmse_{SELECTED_CALIBRATION_PAIRS}'
calibration_block_df['nmse_reduction_pct'] = 100 * (
    1
    - calibration_block_df[selected_column]
    / calibration_block_df[baseline_column]
)
display(
    calibration_block_df.style.format({
        column: '{:.5f}'
        for column in calibration_block_df.columns
        if column.startswith('nmse_')
    } | {'nmse_reduction_pct': '{:.1f}%'})
)

calibration_fitting_plot_df = calibration_fitting_df.copy()
calibration_fitting_plot_df['calibration_budget'] = (
    calibration_fitting_plot_df['calibration_pairs']
    .map(lambda value: f'{int(value):,}')
)
figure, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sns.lineplot(
    data=calibration_fitting_plot_df,
    x='layer',
    y='local_nmse',
    hue='calibration_budget',
    marker='o',
    ax=axes[0],
)
axes[0].set_yscale('log')
axes[0].set(
    title='Per-block local fit',
    xlabel='Transformer block',
    ylabel='Validation NMSE (log scale)',
)
axes[0].legend(title='Pairs per block')
sns.barplot(
    data=calibration_block_df,
    x='layer',
    y='nmse_reduction_pct',
    color=sns.color_palette()[0],
    ax=axes[1],
)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(
    title=(
        f'NMSE change: {baseline_pairs:,} to '
        f'{SELECTED_CALIBRATION_PAIRS:,} pairs'
    ),
    xlabel='Transformer block',
    ylabel='NMSE reduction (%)',
)
figure.tight_layout()
plt.show()

#### Model retraing full budget

In [ ]:
pre_recovery_df = (
    pd.json_normalize(results['sparsity']['model_evaluation'])
    .rename(columns={
        'allocation_selection.teacher_kl': 'selection_kl',
        'allocation_selection.loss': 'selection_loss',
        'allocation_selection.perplexity': 'selection_perplexity',
        'wikitext_validation.loss': 'wikitext_loss',
        'wikitext_validation.perplexity': 'wikitext_perplexity',
    })
)
pre_recovery_df['target_mlp_removal'] = (
    pre_recovery_df['sparsity_key'].astype(float)
)
trajectories = results['recovery']['trajectories']
recovery_history_rows = []
recovery_milestone_rows = []
recovery_evaluation_rows = []

for sparsity_key, trajectory in trajectories.items():
    removal = float(sparsity_key)
    removal_label = f'{removal:.0%} MLP removal'
    pre = pre_recovery_df.loc[
        pre_recovery_df['sparsity_key'] == sparsity_key
    ].iloc[0]
    recovery_evaluation_rows.append({
        'sparsity_key': sparsity_key,
        'target_mlp_removal': removal,
        'removal_label': removal_label,
        'phase': 'Pre-recovery',
        'checkpoint_kind': 'Current',
        'tokens': 0,
        'selection_kl': pre['selection_kl'],
        'wikitext_perplexity': pre['wikitext_perplexity'],
    })
    for point in trajectory['validation_history']:
        recovery_history_rows.append({
            **point,
            'sparsity_key': sparsity_key,
            'target_mlp_removal': removal,
            'removal_label': removal_label,
        })
    for milestone in trajectory['milestones']:
        requested_tokens = int(milestone['requested_tokens'][0])
        current = milestone['current']
        best = milestone['best_under_budget']
        for checkpoint_kind, checkpoint, checkpoint_tokens in (
            ('Current', current, int(milestone['actual_tokens'])),
            ('Best under budget', best, int(best['checkpoint_tokens'])),
        ):
            recovery_milestone_rows.append({
                'sparsity_key': sparsity_key,
                'target_mlp_removal': removal,
                'requested_tokens': requested_tokens,
                'checkpoint_kind': checkpoint_kind,
                'checkpoint_tokens': checkpoint_tokens,
                'recovery_validation_kl': checkpoint[
                    'recovery_validation_kl'
                ],
                'selection_kl': checkpoint[
                    'allocation_selection'
                ]['teacher_kl'],
                'wikitext_perplexity': checkpoint[
                    'wikitext_validation'
                ]['perplexity'],
            })
        recovery_evaluation_rows.append({
            'sparsity_key': sparsity_key,
            'target_mlp_removal': removal,
            'removal_label': removal_label,
            'phase': f'{requested_tokens / 1e6:g}M current',
            'checkpoint_kind': 'Current',
            'tokens': int(milestone['actual_tokens']),
            'selection_kl': current['allocation_selection']['teacher_kl'],
            'wikitext_perplexity': current[
                'wikitext_validation'
            ]['perplexity'],
        })
    final = trajectory['post_recovery_model_evaluation']
    recovery_evaluation_rows.append({
        'sparsity_key': sparsity_key,
        'target_mlp_removal': removal,
        'removal_label': removal_label,
        'phase': 'Best <=100M',
        'checkpoint_kind': 'Best',
        'tokens': int(trajectory['selected_checkpoint']['tokens_seen']),
        'selection_kl': final['allocation_selection']['teacher_kl'],
        'wikitext_perplexity': final[
            'wikitext_validation'
        ]['perplexity'],
    })

recovery_history_df = pd.DataFrame(recovery_history_rows)
recovery_history_df['tokens_millions'] = (
    recovery_history_df['tokens_seen'] / 1e6
)
recovery_milestone_df = pd.DataFrame(recovery_milestone_rows)
recovery_milestone_df['requested_tokens_millions'] = (
    recovery_milestone_df['requested_tokens'] / 1e6
)
recovery_milestone_df['checkpoint_tokens_millions'] = (
    recovery_milestone_df['checkpoint_tokens'] / 1e6
)
recovery_evaluation_df = pd.DataFrame(recovery_evaluation_rows)
recovery_evaluation_df['tokens_millions'] = (
    recovery_evaluation_df['tokens'] / 1e6
)

In [ ]:
recovery_summary_rows = []
for sparsity_key, trajectory in trajectories.items():
    milestones = {
        int(item['requested_tokens'][0]): item
        for item in trajectory['milestones']
    }
    milestone_10m = milestones[10_000_000]['current']
    milestone_100m = milestones[100_000_000]['current']
    final = trajectory['post_recovery_model_evaluation']
    pre = pre_recovery_df.loc[
        pre_recovery_df['sparsity_key'] == sparsity_key
    ].iloc[0]
    best_selection_kl = final['allocation_selection']['teacher_kl']
    best_perplexity = final['wikitext_validation']['perplexity']
    recovery_summary_rows.append({
        'sparsity_key': sparsity_key,
        'target_mlp_removal': float(sparsity_key),
        'pre_selection_kl': pre['selection_kl'],
        'selection_kl_10m': milestone_10m[
            'allocation_selection'
        ]['teacher_kl'],
        'selection_kl_100m_current': milestone_100m[
            'allocation_selection'
        ]['teacher_kl'],
        'best_selection_kl': best_selection_kl,
        'pre_wikitext_perplexity': pre['wikitext_perplexity'],
        'wikitext_perplexity_10m': milestone_10m[
            'wikitext_validation'
        ]['perplexity'],
        'wikitext_perplexity_100m_current': milestone_100m[
            'wikitext_validation'
        ]['perplexity'],
        'best_wikitext_perplexity': best_perplexity,
        'best_checkpoint_tokens_millions': (
            trajectory['selected_checkpoint']['tokens_seen'] / 1e6
        ),
        'best_recovery_validation_kl': trajectory[
            'best_validation_kl'
        ],
        'optimizer_updates': trajectory['optimizer_updates'],
        'recovery_hours': trajectory['elapsed_seconds'] / 3600,
        'selection_kl_reduction_pct': 100 * (
            1 - best_selection_kl / pre['selection_kl']
        ),
        'perplexity_reduction_pct': 100 * (
            1 - best_perplexity / pre['wikitext_perplexity']
        ),
    })
recovery_summary_df = (
    pd.DataFrame(recovery_summary_rows)
    .sort_values('target_mlp_removal')
    .reset_index(drop=True)
)

display(
    recovery_summary_df.style.format({
        'target_mlp_removal': '{:.0%}',
        'pre_selection_kl': '{:.6f}',
        'selection_kl_10m': '{:.6f}',
        'selection_kl_100m_current': '{:.6f}',
        'best_selection_kl': '{:.6f}',
        'pre_wikitext_perplexity': '{:.3f}',
        'wikitext_perplexity_10m': '{:.3f}',
        'wikitext_perplexity_100m_current': '{:.3f}',
        'best_wikitext_perplexity': '{:.3f}',
        'best_checkpoint_tokens_millions': '{:.1f}M',
        'best_recovery_validation_kl': '{:.6f}',
        'optimizer_updates': '{:,.0f}',
        'recovery_hours': '{:.2f}',
        'selection_kl_reduction_pct': '{:.1f}%',
        'perplexity_reduction_pct': '{:.1f}%',
    })
)
display(
    recovery_milestone_df[[
        'target_mlp_removal',
        'requested_tokens_millions',
        'checkpoint_kind',
        'checkpoint_tokens_millions',
        'recovery_validation_kl',
        'selection_kl',
        'wikitext_perplexity',
    ]].style.format({
        'target_mlp_removal': '{:.0%}',
        'requested_tokens_millions': '{:.0f}M',
        'checkpoint_tokens_millions': '{:.1f}M',
        'recovery_validation_kl': '{:.6f}',
        'selection_kl': '{:.6f}',
        'wikitext_perplexity': '{:.3f}',
    })
)

In [ ]:
current_recovery_evaluation_df = recovery_evaluation_df.query(
    "checkpoint_kind == 'Current'"
)
best_recovery_evaluation_df = recovery_evaluation_df.query(
    "checkpoint_kind == 'Best'"
)
figure, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.lineplot(
    data=recovery_history_df,
    x='tokens_millions',
    y='recovery_validation_kl',
    hue='removal_label',
    hue_order=SPARSITY_LABEL_ORDER,
    palette=SPARSITY_PALETTE,
    ax=axes[0],
)
for _, row in recovery_summary_df.iterrows():
    removal_label = f"{row['target_mlp_removal']:.0%} MLP removal"
    axes[0].scatter(
        row['best_checkpoint_tokens_millions'],
        row['best_recovery_validation_kl'],
        color=SPARSITY_PALETTE[removal_label],
        marker='*',
        s=180,
        edgecolor='black',
        linewidth=0.6,
        zorder=5,
    )
axes[0].set(
    title='Continuous recovery trajectory',
    xlabel='Recovery tokens (millions)',
    ylabel='Fixed validation KL',
)

for axis, metric, title, ylabel in (
    (axes[1], 'selection_kl', 'Model KL at full evaluations', 'C4 teacher KL'),
    (
        axes[2],
        'wikitext_perplexity',
        'Perplexity at full evaluations',
        'WikiText-2 perplexity',
    ),
):
    sns.lineplot(
        data=current_recovery_evaluation_df,
        x='tokens_millions',
        y=metric,
        hue='removal_label',
        hue_order=SPARSITY_LABEL_ORDER,
        palette=SPARSITY_PALETTE,
        marker='o',
        ax=axis,
    )
    for _, row in best_recovery_evaluation_df.iterrows():
        axis.scatter(
            row['tokens_millions'],
            row[metric],
            color=SPARSITY_PALETTE[row['removal_label']],
            marker='*',
            s=180,
            edgecolor='black',
            linewidth=0.6,
            zorder=5,
        )
    axis.set(title=title, xlabel='Recovery tokens (millions)', ylabel=ylabel)
axes[2].axhline(
    DENSE_PERPLEXITY,
    color='black',
    linestyle='--',
    label='Dense model',
)
axes[2].legend()
figure.suptitle('100M-token replacement-only recovery')
figure.tight_layout()
plt.show()

#### Global sparsity sweep

In [ ]:
allocation_df = pd.DataFrame(results['sparsity']['allocation'])
allocation_df['target_mlp_removal'] = (
    allocation_df['sparsity_key'].astype(float)
)
allocation_df['removal_label'] = allocation_df[
    'target_mlp_removal'
].map(lambda value: f'{value:.0%} MLP removal')
allocation_summary_df = pd.DataFrame(
    results['sparsity']['allocation_summary']
)
allocation_summary_df['target_mlp_removal'] = (
    allocation_summary_df['sparsity_key'].astype(float)
)
allocation_summary_df['model_parameters'] = (
    DENSE_PARAMETERS
    - allocation_summary_df['realized_removed_parameters']
).astype(int)
allocation_summary_df['model_parameter_reduction_pct'] = (
    100 * allocation_summary_df['realized_whole_model_removal']
)
allocation_summary_df['mlp_parameter_reduction_pct'] = (
    100 * allocation_summary_df['realized_eligible_mlp_removal']
)

selected_half_fitting_df = calibration_fitting_df.loc[
    calibration_fitting_df['calibration_pairs']
    == SELECTED_CALIBRATION_PAIRS
].assign(sparsity_key='0.5', requested_mlp_removal=0.5)
sparsity_fitting_df = pd.concat(
    [
        selected_half_fitting_df,
        pd.DataFrame(results['sparsity']['operator_fitting']),
    ],
    ignore_index=True,
)
sparsity_fitting_df['target_mlp_removal'] = (
    sparsity_fitting_df['sparsity_key'].astype(float)
)
sparsity_fitting_df['removal_label'] = sparsity_fitting_df[
    'target_mlp_removal'
].map(lambda value: f'{value:.0%} MLP removal')
sparsity_local_summary_df = (
    sparsity_fitting_df
    .groupby('sparsity_key', as_index=False)
    .agg(
        mean_nmse=('local_nmse', 'mean'),
        worst_nmse=('local_nmse', 'max'),
        mean_cosine=('local_cosine', 'mean'),
        total_fit_minutes=(
            'fit_seconds', lambda values: values.sum() / 60
        ),
    )
)

In [ ]:
sparsity_summary_df = (
    allocation_summary_df
    .merge(
        sparsity_local_summary_df,
        on='sparsity_key',
        validate='one_to_one',
    )
    .merge(
        recovery_summary_df.drop(columns='target_mlp_removal'),
        on='sparsity_key',
        validate='one_to_one',
    )
    .sort_values('target_mlp_removal')
    .reset_index(drop=True)
)
display(
    sparsity_summary_df[[
        'target_mlp_removal',
        'mlp_parameter_reduction_pct',
        'model_parameter_reduction_pct',
        'model_parameters',
        'minimum_retention',
        'maximum_retention',
        'mean_nmse',
        'worst_nmse',
        'pre_selection_kl',
        'best_selection_kl',
        'pre_wikitext_perplexity',
        'best_wikitext_perplexity',
        'selection_kl_reduction_pct',
        'perplexity_reduction_pct',
        'best_checkpoint_tokens_millions',
    ]].style.format({
        'target_mlp_removal': '{:.0%}',
        'mlp_parameter_reduction_pct': '{:.2f}%',
        'model_parameter_reduction_pct': '{:.2f}%',
        'model_parameters': '{:,.0f}',
        'minimum_retention': '{:.2%}',
        'maximum_retention': '{:.2%}',
        'mean_nmse': '{:.5f}',
        'worst_nmse': '{:.5f}',
        'pre_selection_kl': '{:.6f}',
        'best_selection_kl': '{:.6f}',
        'pre_wikitext_perplexity': '{:.3f}',
        'best_wikitext_perplexity': '{:.3f}',
        'selection_kl_reduction_pct': '{:.1f}%',
        'perplexity_reduction_pct': '{:.1f}%',
        'best_checkpoint_tokens_millions': '{:.1f}M',
    })
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(
    data=allocation_df,
    x='layer',
    y='replacement_width_ratio',
    hue='removal_label',
    hue_order=SPARSITY_LABEL_ORDER,
    palette=SPARSITY_PALETTE,
    marker='o',
    ax=axes[0],
)
axes[0].set(
    title='Fixed-policy width allocation',
    xlabel='Transformer block',
    ylabel='Retained SwiGLU width',
)
axes[0].legend(title='Budget')
sns.lineplot(
    data=sparsity_fitting_df,
    x='layer',
    y='local_nmse',
    hue='removal_label',
    hue_order=SPARSITY_LABEL_ORDER,
    palette=SPARSITY_PALETTE,
    marker='o',
    ax=axes[1],
)
axes[1].set_yscale('log')
axes[1].set(
    title='Local fit at allocated widths',
    xlabel='Transformer block',
    ylabel='Validation NMSE (log scale)',
)
axes[1].legend(title='Budget')
figure.tight_layout()
plt.show()

In [ ]:
quality_curve_df = recovery_evaluation_df[
    recovery_evaluation_df['phase'].isin([
        'Pre-recovery',
        '10M current',
        '100M current',
        'Best <=100M',
    ])
].merge(
    allocation_summary_df[[
        'sparsity_key', 'model_parameter_reduction_pct'
    ]],
    on='sparsity_key',
    validate='many_to_one',
)
QUALITY_PHASE_ORDER = [
    'Pre-recovery', '10M current', '100M current', 'Best <=100M'
]
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, metric, title, ylabel in (
    (
        axes[0],
        'selection_kl',
        'Compression versus teacher KL',
        'C4 teacher KL',
    ),
    (
        axes[1],
        'wikitext_perplexity',
        'Compression versus perplexity',
        'WikiText-2 perplexity',
    ),
):
    sns.lineplot(
        data=quality_curve_df,
        x='model_parameter_reduction_pct',
        y=metric,
        hue='phase',
        hue_order=QUALITY_PHASE_ORDER,
        style='phase',
        markers=True,
        dashes=False,
        ax=axis,
    )
    axis.set(
        title=title,
        xlabel='Whole-model parameter reduction (%)',
        ylabel=ylabel,
    )
axes[1].axhline(
    DENSE_PERPLEXITY, color='black', linestyle='--', label='Dense model'
)
axes[1].legend()
figure.tight_layout()
plt.show()

artifact

In [ ]:
configuration_table_df = pd.DataFrame([
    ('Workflow', artifact['workflow']),
    ('Status', artifact['status']),
    ('Model', configuration['model']['model_id']),
    ('Model revision', configuration['model']['revision']),
    ('Winning allocation', configuration['references']['winning_policy']),
    ('Initialization', configuration['local_fitting']['initialization']),
    ('Calibration candidates', ', '.join(
        f'{value:,}' for value in CALIBRATION_PAIR_ORDER
    )),
    ('Selected calibration pairs', f'{SELECTED_CALIBRATION_PAIRS:,}'),
    ('Operator batch size', configuration['local_fitting']['batch_size']),
    (
        'Operator maximum epochs',
        configuration['local_fitting']['max_epochs'],
    ),
    (
        'Operator parameter dtype',
        configuration['local_fitting']['parameter_dtype'],
    ),
    (
        'Recovery tokens per model',
        f"{configuration['recovery']['target_tokens_per_model']:,}",
    ),
    ('Recovery milestones', ', '.join(
        f'{value:,}' for value in configuration['recovery']['milestone_tokens']
    )),
    (
        'Recovery effective batch tokens',
        configuration['recovery']['effective_batch_tokens'],
    ),
    (
        'Recovery parameter dtype',
        configuration['recovery']['replacement_parameter_dtype'],
    ),
], columns=['setting', 'value'])
display(configuration_table_df)

In [ ]:
runtime_df = pd.DataFrame(results['runtime'])
runtime_df['minutes'] = runtime_df['seconds'] / 60
runtime_df['hours'] = runtime_df['seconds'] / 3600
resource_rows = []
for sparsity_key, trajectory in trajectories.items():
    first_step = trajectory['first_step']
    memory = trajectory['memory']
    resource_rows.append({
        'target_mlp_removal': float(sparsity_key),
        'tokens_seen': trajectory['tokens_seen'],
        'optimizer_updates': trajectory['optimizer_updates'],
        'elapsed_hours': trajectory['elapsed_seconds'] / 3600,
        'peak_ram_gib': memory['peak_ram_gib'],
        'peak_vram_gib': memory['peak_vram_gib'],
        'first_gradient_l2_norm': first_step['gradient_l2_norm'],
        'first_parameter_max_abs_update': first_step[
            'first_parameter_max_abs_update'
        ],
        'parameter_dtype': first_step['replacement_parameter_dtype'],
        'optimizer_state_dtypes': ', '.join(
            first_step['optimizer_state_dtypes']
        ),
    })
resource_df = pd.DataFrame(resource_rows).sort_values(
    'target_mlp_removal'
)
environment_df = pd.DataFrame([{
    'completed_at_utc': artifact['completed_at_utc'],
    'python': artifact['environment']['python'],
    'torch': artifact['environment']['packages']['torch'],
    'transformers': artifact['environment']['packages']['transformers'],
    'cuda_runtime': artifact['environment']['cuda_runtime'],
    'gpu': artifact['environment']['gpu'],
    'total_recovery_tokens': results['recovery'][
        'total_recovery_tokens'
    ],
    'artifact_path': str(SWIGLU_3_ARTIFACT_PATH),
}])
provenance_df = pd.DataFrame([
    {
        'source': source,
        'path': path,
        'sha256': artifact['provenance']['source_sha256'][source],
    }
    for source, path in artifact['provenance']['source_paths'].items()
])

display(runtime_df.style.format({
    'seconds': '{:,.1f}', 'minutes': '{:,.1f}', 'hours': '{:.2f}'
}))
display(resource_df.style.format({
    'target_mlp_removal': '{:.0%}',
    'tokens_seen': '{:,.0f}',
    'optimizer_updates': '{:,.0f}',
    'elapsed_hours': '{:.2f}',
    'peak_ram_gib': '{:.2f}',
    'peak_vram_gib': '{:.2f}',
    'first_gradient_l2_norm': '{:.6f}',
    'first_parameter_max_abs_update': '{:.2e}',
}))
display(environment_df)
display(provenance_df)